2-D Daubechies Wavelets
=======================

*Important:* Please read the [installation page](http://gpeyre.github.io/numerical-tours/installation_python/) for details about how to install the toolboxes.
$\newcommand{\dotp}[2]{\langle #1, #2 \rangle}$
$\newcommand{\enscond}[2]{\lbrace #1, #2 \rbrace}$
$\newcommand{\pd}[2]{ \frac{ \partial #1}{\partial #2} }$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\umax}[1]{\underset{#1}{\max}\;}$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\uargmin}[1]{\underset{#1}{argmin}\;}$
$\newcommand{\norm}[1]{\|#1\|}$
$\newcommand{\abs}[1]{\left|#1\right|}$
$\newcommand{\choice}[1]{ \left\{  \begin{array}{l} #1 \end{array} \right. }$
$\newcommand{\pa}[1]{\left(#1\right)}$
$\newcommand{\diag}[1]{{diag}\left( #1 \right)}$
$\newcommand{\qandq}{\quad\text{and}\quad}$
$\newcommand{\qwhereq}{\quad\text{where}\quad}$
$\newcommand{\qifq}{ \quad \text{if} \quad }$
$\newcommand{\qarrq}{ \quad \Longrightarrow \quad }$
$\newcommand{\ZZ}{\mathbb{Z}}$
$\newcommand{\CC}{\mathbb{C}}$
$\newcommand{\RR}{\mathbb{R}}$
$\newcommand{\EE}{\mathbb{E}}$
$\newcommand{\Zz}{\mathcal{Z}}$
$\newcommand{\Ww}{\mathcal{W}}$
$\newcommand{\Vv}{\mathcal{V}}$
$\newcommand{\Nn}{\mathcal{N}}$
$\newcommand{\NN}{\mathcal{N}}$
$\newcommand{\Hh}{\mathcal{H}}$
$\newcommand{\Bb}{\mathcal{B}}$
$\newcommand{\Ee}{\mathcal{E}}$
$\newcommand{\Cc}{\mathcal{C}}$
$\newcommand{\Gg}{\mathcal{G}}$
$\newcommand{\Ss}{\mathcal{S}}$
$\newcommand{\Pp}{\mathcal{P}}$
$\newcommand{\Ff}{\mathcal{F}}$
$\newcommand{\Xx}{\mathcal{X}}$
$\newcommand{\Mm}{\mathcal{M}}$
$\newcommand{\Ii}{\mathcal{I}}$
$\newcommand{\Dd}{\mathcal{D}}$
$\newcommand{\Ll}{\mathcal{L}}$
$\newcommand{\Tt}{\mathcal{T}}$
$\newcommand{\si}{\sigma}$
$\newcommand{\al}{\alpha}$
$\newcommand{\la}{\lambda}$
$\newcommand{\ga}{\gamma}$
$\newcommand{\Ga}{\Gamma}$
$\newcommand{\La}{\Lambda}$
$\newcommand{\si}{\sigma}$
$\newcommand{\Si}{\Sigma}$
$\newcommand{\be}{\beta}$
$\newcommand{\de}{\delta}$
$\newcommand{\De}{\Delta}$
$\newcommand{\phi}{\varphi}$
$\newcommand{\th}{\theta}$
$\newcommand{\om}{\omega}$
$\newcommand{\Om}{\Omega}$

This numerical tour explores 2-D multiresolution analysis
with Daubchies wavelet transform.

In [ ]:
from __future__ import division
import nt_toolbox as nt
from nt_solutions import wavelet_4_daubechies2d as solutions
%matplotlib inline
%load_ext autoreload
%autoreload 2

Wavelets Filters
----------------
The 2-D wavelet transform of a continuous image $f(x)$ computes the set
of inner products
$$ d_j^k[n] = \dotp{f}{\psi_{j,n}^k} $$
for scales $ j \in \ZZ
$, position $ n \in \ZZ^2 $ and orientation $ k \in \{H,V,D\} $.


The wavelet atoms are defined by scaling and translating three mother
atoms $ \{\psi^H,\psi^V,\psi^D\} $:
$$ \psi_{j,n}^k(x) = \frac{1}{2^j}\psi^k\pa{\frac{x-2^j n}{2^j}}  $$
These oriented wavelets are defined by a tensor product of a 1-D wavelet
function $\psi(t)$ and a 1-D scaling function $\phi(t)$
$$ \psi^H(x)=\phi(x_1)\psi(x_2), \quad  \psi^V(x)=\psi(x_1)\phi(x_2)
\qandq \psi^D(x)=\psi(x_1)\psi(x_2).$$


The fast wavelet transform algorithm does not make use of the wavelet and scaling functions,
but of the filters $h$ and $g$ that caracterize their interaction:
$$ g[n] = \frac{1}{\sqrt{2}}\dotp{\psi(t/2)}{\phi(t-n)}
\qandq h[n] = \frac{1}{\sqrt{2}}\dotp{\phi(t/2)}{\phi(t-n)}. $$


The simplest filters are the Haar filters
$$ h = [1, 1]/\sqrt{2} \qandq g = [-1, 1]/\sqrt{2}. $$


Daubechies wavelets extends the haar wavelets by using longer
filters, that produce smoother scaling functions and wavelets.
Furthermore, the larger the size $p=2k$ of the filter, the higher is the number
$k$ of vanishing moment.


A high number of vanishing moments allows to better compress regular
parts of the signal. However, increasing the number of vanishing moments
also inceases the size of the support of the wavelets, wich can be
problematic in part where the signal is singular (for instance
discontinuous).


Choosing the _best_ wavelet, and thus choosing $k$, that is adapted to a
given class of signals, thus corresponds to
a tradeoff between efficiency in regular and singular parts.


* The filter with $k=1$ vanishing moments corresponds to the Haar filter.
* The filter with $k=2$ vanishing moments corresponds to the famous |D4| wavelet, which compresses perfectly linear signals.
* The filter with $k=3$ vanishing moments compresses perfectly quadratic signals.


Set the support size.
To begin, we select the D4 filter.

In [ ]:
p = 4

Create the low pass filter $h$ and the high pass $g$. We add a zero to ensure that it has a odd
length. Note that the central value of $h$ corresponds to the 0 position.

In [ ]:
[h, g] = compute_wavelet_filter('Daubechies', p)

Note that the high pass filter $g$ is computed directly from the low
pass filter as:
$$g[n] = (-1)^{1-n}h[1-n]$$


Display.

In [ ]:
disp(['h filter = [' num2str(h) ']'])
disp(['g filter = [' num2str(g) ']'])

Up and Down Filtering
---------------------
The basic wavelet operation is low/high filtering, followed by down
sampling.


Starting from some 1-D signal $f \in \RR^N$, one thus compute the
low pass signal $a \in \RR^{N/2}$ and the high pass
signal $d \in \RR^{N/2}$ as
$$ a = (f \star h) \downarrow 2 \qandq
d = (f \star g) \downarrow 2$$
where the sub-sampling is defined as
$$ (u \downarrow 2)[k] = u[2k]. $$


Create a random signal $f \in \RR^N$.

In [ ]:
N = 256
f = rand(N, 1)

Low/High pass filtering followed by sub-sampling.

In [ ]:
a = subsampling(cconvol(f, h))
d = subsampling(cconvol(f, g))

For orthogonal filters, the reverse of this process is its dual
(aka its transpose), which is upsampling followed by low/high pass
filtering with the reversed filters and summing:
$$ (a \uparrow h) \star \tilde h + (d \uparrow g) \star \tilde g = f $$
where $\tilde h[n]=h[-n]$ (computed modulo $N$) and
$ (u \uparrow 2)[2n]=u[n] $ and  $ (u \uparrow 2)[2n+1]=0 $.


Up-sampling followed by filtering.

In [ ]:
f1 =  cconvol(upsampling(a), reverse(h)) + cconvol(upsampling(d), reverse(g))

Check that we really recover the same signal.

In [ ]:
disp(strcat((['Error |f-f1|/ |f| = ' num2str(norm(f-f1)/ norm(f))])))

Forward 2-D Wavelet transform
-----------------------------
The set of wavelet coefficients are computed with a fast algorithm that
exploit the embedding of the approximation spaces $V_j$ spanned by the
scaling function $ \{ \phi_{j,n} \}_n $ defined as
$$ \phi_{j,n}(x) = \frac{1}{2^j}\phi^0\pa{\frac{x-2^j n}{2^j}}
\qwhereq \phi^0(x)=\phi(x_1)\phi(x_2). $$


The wavelet transform of $f$ is computed by using intermediate discretized low
resolution images obtained by projection on the spaces $V_j$:
$$ a_j[n] = \dotp{f}{\phi_{j,n}}. $$


First we load an image of $N= n \times n$ pixels.

In [ ]:
n = 256
name = 'hibiscus'
f = load_image(name, n)
f = rescale(sum(f, 3))

The algorithm starts at the coarsest scale $ j=\log_2(n)-1 $

In [ ]:
j = log2(n)-1

The first step of the algorithm perform filtering/downsampling in the
horizontal direction.


$$ \tilde a_{j-1} = (a_j \star^H h) \downarrow^{2,H}  \qandq
   \tilde d_{j-1} = (a_j \star^H g) \downarrow^{2,H}$$


Here, the operator $\star^H$ and $\downarrow^{2,H}$
are defined by applying $\star$ and $\downarrow^2$
to each column of the matrix.


The second step computes the filtering/downsampling in the vertical
direction.


$$ a_{j-1}   = (\tilde a_j \star^V h) \downarrow^{2,V}  \qandq
   d_{j-1}^V = (\tilde a_j \star^V g) \downarrow^{2,V},$$
$$ d_{j-1}^H = (\tilde d_j \star^V h) \downarrow^{2,V}  \qandq
   d_{j-1}^D = (\tilde d_j \star^V g) \downarrow^{2,V}.$$



A wavelet transform is
computed by iterating high pass and loss pass filterings with |h| and |g|, followed by sub-samplings.
Since we are in 2-D, we need to compute these filterings+subsamplings
in the horizontal and then in the vertical direction (or
in the reverse order, it does not mind).



Initialize the transformed coefficients as the image itself and set the
initial scale as the maximum one.
|fW| will be iteratively transformated and will contains the
coefficients.

In [ ]:
fW = f

Select the sub-part of the image to transform.

In [ ]:
A = fW(1: 2^(j + 1), 1: 2^(j + 1))

Apply high and low filtering+subsampling in the vertical direction (1st ooordinate),
to get coarse and details.

In [ ]:
Coarse = subsampling(cconvol(A, h, 1), 1)
Detail = subsampling(cconvol(A, g, 1), 1)

_Note:_ |subsamplling(A,1)| is equivalent to |A(1:2:end,:)| and
|subsamplling(A,2)| is equivalent to |A(:,1:2:end)|.


Concatenate them in the vertical direction to get the result.

In [ ]:
A = cat3(1, Coarse, Detail)

Display the result of the vertical transform.

In [ ]:
imageplot(f, 'Original imge', 1, 2, 1)
imageplot(A, 'Vertical transform', 1, 2, 2)

Apply high and low filtering+subsampling in the horizontal direction (2nd ooordinate),
to get coarse and details.

In [ ]:
Coarse = subsampling(cconvol(A, h, 2), 2)
Detail = subsampling(cconvol(A, g, 2), 2)

Concatenate them in the horizontal direction to get the result.

In [ ]:
A = cat3(2, Coarse, Detail)

Assign the transformed data.

In [ ]:
fW(1: 2^(j + 1), 1: 2^(j + 1)) = A

Display the result of the horizontal transform.

In [ ]:
imageplot(f, 'Original image', 1, 2, 1)
subplot(1, 2, 2)
plot_wavelet(fW, log2(n)-1); title('Transformed')

__Exercise 1__

Implement a full wavelet transform that extract iteratively wavelet
coefficients, by repeating these steps. Take care of choosing the
correct number of steps.

In [ ]:
solutions.exo1()

In [ ]:
## Insert your code here.

Check for orthogonality of the transform (conservation of energy).

In [ ]:
disp(strcat(['Energy of the signal       = ' num2str(norm(f(: )).^2)]))
disp(strcat(['Energy of the coefficients = ' num2str(norm(fW(: )).^2)]))

Display the wavelet coefficients.

In [ ]:
subplot(1, 2, 1)
imageplot(f); title('Original')
subplot(1, 2, 2)
plot_wavelet(fW, 1); title('Transformed')

Inverse 2-D Wavelet transform.
------------------------------
Inversing the wavelet transform means retrieving a signal |f1| from the
coefficients |fW|. If |fW| are exactely the coefficients of |f|, then
|f=f1| up to machine precision.


Initialize the image to recover |f1| as the transformed coefficient, and
select the smallest possible scale.

In [ ]:
f1 = fW
j = 0

Select the sub-coefficient to transform.

In [ ]:
A = f1(1: 2^(j + 1), 1: 2^(j + 1))

Retrieve coarse and detail coefficients in the vertical direction (you
can begin by the other direction, this has no importance).

In [ ]:
Coarse = A(1: 2^j, : )
Detail = A(2^j + 1: 2^(j + 1), : )

Undo the transform by up-sampling and then dual filtering.

In [ ]:
Coarse = cconvol(upsampling(Coarse, 1), reverse(h), 1)
Detail = cconvol(upsampling(Detail, 1), reverse(g), 1)

Recover the coefficient by summing.

In [ ]:
A = Coarse + Detail

Retrieve coarse and detail coefficients in the vertical direction (you
can begin by the other direction, this has no importance).

In [ ]:
Coarse = A(: , 1: 2^j)
Detail = A(: , 2^j + 1: 2^(j + 1))

Undo the transform by up-sampling and then dual filtering.

In [ ]:
Coarse = cconvol(upsampling(Coarse, 2), reverse(h), 2)
Detail = cconvol(upsampling(Detail, 2), reverse(g), 2)

Recover the coefficient by summing.

In [ ]:
A = Coarse + Detail

Assign the result.

In [ ]:
f1(1: 2^(j + 1), 1: 2^(j + 1)) = A

__Exercise 2__

Write the inverse wavelet transform that computes |f1| from the
coefficients |fW|. Compare |f1| with |f|.

In [ ]:
solutions.exo2()

In [ ]:
## Insert your code here.

Check that we recover exactly the original image.

In [ ]:
disp(strcat((['Error |f-f1|/ |f| = ' num2str(norm(f(: )-f1(: ))/ norm(f(: )))])))

Linear 2-D Wavelet Approximation
--------------------------------
Linear approximation is performed by setting to zero the fine scale wawelets coefficients
and then performing the inverse wavelet transform.


Here we keep only 1/16 of the wavelet coefficient, thus calculating an |m|
term approximation with |m=n^2/16|.

In [ ]:
eta = 4
fWLin = zeros(n, n)
fWLin(1: n/ eta, 1: n/ eta) = fW(1: n/ eta, 1: n/ eta)

__Exercise 3__

Compute and display the linear approximation |fLin| obtained from the
coefficients |fWLin| by inverse wavelet transform.

In [ ]:
solutions.exo3()

In [ ]:
## Insert your code here.

Non-Linear 2-D Wavelet Approximation
------------------------------------
A non-linear |m|-term approximation is obtained by keeping only the |m|
largest coefficients, which creates the smallest possible error.


Removing the smallest coefficient, to keep the |m|-largest, is
equivalently obtainedby thresholding the coefficients to
set to 0 the smallest coefficients.


First select a threshold value (the largest the threshold, the more
agressive the approximation).

In [ ]:
T = .2

Then set to 0 coefficients with magnitude below the threshold.

In [ ]:
fWT = fW .* (abs(fW) >T)

Display thresholded coefficients.

In [ ]:
subplot(1, 2, 1)
plot_wavelet(fW); axis('tight'); title('Original coefficients')
subplot(1, 2, 2)
plot_wavelet(fWT); axis('tight'); title('Thresholded coefficients')

__Exercise 4__

Find the thresholds |T| so that the number |m| of remaining coefficients in
|fWT| are |m=n^2/16|. Use this threshold to compute |fWT| and then display
the corresponding approximation |Mnlin| of |f|. Compare this result with
the linear approximation.
umber of kept coefficients
ompute the threshold T
elect threshold
nverse transform
nverse
isplay

In [ ]:
solutions.exo4()

In [ ]:
## Insert your code here.

__Exercise 5__

Try with
Different kind of wavelets, with an increasing number of vanishing
moments.

In [ ]:
solutions.exo5()

In [ ]:
## Insert your code here.

Separable 2-D Wavelet Transform
-------------------------------
A forward wavelet transform is obtained by applying the 1D wavelet
transform to

__Exercise 6__

Implement the foward separable transform.
Wavelet transformm in 1D each column |f(:,i)| to obtain coefficients |fWSep(:,i)|.
Then re-transform each row |fWSep(i,:)'|, and store the result in |fW(i,:)'|.

In [ ]:
solutions.exo6()

In [ ]:
## Insert your code here.

Display the result.

In [ ]:
subplot(1, 2, 1)
opt.separable = 0
plot_wavelet(fW, 1, opt)
title('Isotropic wavelets')
subplot(1, 2, 2)
opt.separable = 1
plot_wavelet(fWSep, 1, opt)
title('Separable wavelets')

__Exercise 7__

Implement the backward separable transform to recover an image |f1|
from the coefficients |fWSep|, which backward transform each row and then each columns.

In [ ]:
solutions.exo7()

In [ ]:
## Insert your code here.

Check that we recover exactly the original image.

In [ ]:
disp(strcat((['Error |f-f1|/ |f| = ' num2str(norm(f(: )-f1(: ))/ norm(f(: )))])))

__Exercise 8__

Perform |m=n^2/16|-terms best approximation with separable wavelet transform.
Compare the result with isotropic wavelet approximation.
isplay

In [ ]:
solutions.exo8()

In [ ]:
## Insert your code here.

The Shape of a Wavelet
----------------------
An isotropic wavelet coefficient |fW[k]| corresponds to an inner product
|<f,psi_{j,p}^s>| where |k| depends on the scale |j| and the position |p| and the orientation
|s| (horizontal, vertical or diagonal).


The wavelet image  |f1 = psi_{j,p}| is computed by applying the inverse wavelet
transform to |fW| where |fW[k]=1| and |fW[l]=0| for |l \neq k|.

__Exercise 9__

Compute wavelets at several scales and orientation.
Here we show only horizontal wavelets, in 2-D.

In [ ]:
solutions.exo9()

In [ ]:
## Insert your code here.

__Exercise 10__

Display Daubechies wavelets with different orientation, for different number of VM.

In [ ]:
solutions.exo10()

In [ ]:
## Insert your code here.